# Vinamilk OCR benchmark (Kaggle GPU or CPU)

Run Tesseract, PaddleOCR, EasyOCR, or Surya against ten manually transcribed Vinamilk report regions from five official PDFs. The v7 set includes Vietnamese report pages and English scans; the page references are versioned and hash-checked. Attach the private dataset `ltrngnhn/vinamilk-ocr-v7-benchmark-assets` and enable **Internet**. For GPU runs, enable a Kaggle GPU and keep `REQUIRE_GPU = True`; for an intentional CPU run, disable the accelerator and set `REQUIRE_GPU = False`. PaddleOCR currently installs the CPU wheel. Record device and do not compare throughput across unlike hardware.

The dataset contains evaluator code, reviewed references, and only the five public Vinamilk source PDFs needed to render the labeled regions; it contains no credentials or private uploads. Compare results only when the v7 label and source hashes match. The Surya option pins version 0.17.1 and must be labeled Surya v1; Surya 2 has a different server-backed architecture.


In [ ]:
from pathlib import Path
import shutil, subprocess, sys

input_root = Path("/kaggle/input")
for bundle in [*input_root.glob("*/benchmark_project.zip"), *input_root.glob("datasets/*/*/benchmark_project.zip")]:
    extracted = Path("/kaggle/working/benchmark-assets/project")
    shutil.rmtree(extracted.parent, ignore_errors=True)
    shutil.unpack_archive(str(bundle), str(extracted.parent))
project_candidates = {
    *input_root.glob("*/project"),
    *input_root.glob("datasets/*/*/project"),
    *Path("/kaggle/working/benchmark-assets").glob("project"),
    *(entry.parent.parent for entry in input_root.rglob("evaluate_ocr.py") if entry.parent.name == "scripts"),
}
assets_candidates = sorted(
    project for project in project_candidates
    if (project / "scripts" / "evaluate_ocr.py").is_file()
)
if len(assets_candidates) != 1:
    raise FileNotFoundError(
        "Expected exactly one attached vinamilk-ocr-v7-benchmark-assets/project; "
        f"found: {assets_candidates}"
    )
assets = assets_candidates[0]
repo = Path("/kaggle/working/enterprise-knowledge-ai-agent")
if repo.exists():
    shutil.rmtree(repo)
shutil.copytree(assets, repo)
%cd /kaggle/working/enterprise-knowledge-ai-agent


In [ ]:
ENGINE = "tesseract"  # Change to paddleocr, easyocr, or surya for separate kernels.
REQUIRE_GPU = False  # Tesseract is CPU-only; set True for EasyOCR/Surya GPU runs.
DPI = 180
TESSERACT_PSM = 3  # Use 11 for the paired sparse-text layout comparison.


In [ ]:
import subprocess, sys

# Install one engine only, avoiding incompatible OCR framework combinations.
subprocess.run([sys.executable, "-m", "pip", "install", "PyMuPDF==1.27.2.3", "Pillow==12.1.0"], check=True)
if ENGINE == "paddleocr":
    subprocess.run([sys.executable, "-m", "pip", "--default-timeout=300", "--retries=5", "install", "paddlepaddle==3.3.0", "-i", "https://www.paddlepaddle.org.cn/packages/stable/cpu/"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "paddleocr==3.7.0"], check=True)
elif ENGINE == "easyocr":
    subprocess.run([sys.executable, "-m", "pip", "install", "easyocr==1.7.2"], check=True)
elif ENGINE == "surya":
    subprocess.run([sys.executable, "-m", "pip", "install", "surya-ocr==0.17.1", "transformers==4.57.3"], check=True)
elif ENGINE == "tesseract":
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "tesseract-ocr", "tesseract-ocr-vie"], check=True)
else:
    raise ValueError(f"Unknown ENGINE: {ENGINE}")


In [ ]:
import os

os.environ["HF_HOME"] = "/kaggle/working/model-cache/huggingface"
os.environ["TORCH_HOME"] = "/kaggle/working/model-cache/torch"
if ENGINE == "surya":
    # Fit Surya's detection/recognition batches within Kaggle's 15 GB Tesla T4.
    os.environ["DETECTOR_BATCH_SIZE"] = "16"
    os.environ["RECOGNITION_BATCH_SIZE"] = "128"
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
import torch
cuda_available = torch.cuda.is_available()
print("torch", torch.__version__, "CUDA", cuda_available)
if ENGINE in {"easyocr", "surya"} and REQUIRE_GPU and not cuda_available:
    raise RuntimeError("GPU requested for this OCR engine but CUDA is unavailable; restart with a Kaggle GPU accelerator.")
if ENGINE == "paddleocr":
    import paddle
    print("paddle", paddle.__version__, "CUDA build", paddle.is_compiled_with_cuda(), "device", paddle.device.get_device())


In [ ]:
for document_id in ("vnm-ar-2024-vi", "vnm-sr-2024-en", "vnm-ar-2020-vi", "vnm-charter-2026-en", "vnm-cgrules-2026-en"):
    matches = list((Path("data/raw/vietnam_enterprises/vnm")).glob(f"{document_id}.pdf"))
    if len(matches) != 1:
        raise FileNotFoundError(f"Missing benchmark source PDF: {document_id}.pdf")
output = Path("/kaggle/working") / f"vinamilk-ocr-{ENGINE}-v7.json"
command = [sys.executable, "scripts/evaluate_ocr.py", "--dataset", "data/evaluation/vinamilk_ocr_ground_truth_v7.json", "--engines", ENGINE, "--dpi", str(DPI), "--output", str(output)]
if ENGINE == "tesseract":
    command.extend(["--tesseract-psm", str(TESSERACT_PSM)])
subprocess.run(command, check=True)
print(f"Download the result from Kaggle Outputs: {output}")


## Limitations

The ten reviewed regions include six v6 regions and four additional English physical-scan regions. They span five PDFs; they do not yet provide representative Vietnamese physical scans or cell-coordinate labels. OCR text recognition and character/word error rates do not establish layout or table-structure accuracy. Preserve the generated JSON, package/model versions, GPU name, and source/render hashes with each result.
